# Notebook Databricks — Agregação Gold

**Objetivo:** ler `lakehouse_catalog.silver.livros`, construir as dimensões e a tabela fato do modelo estrela, e gravar o resultado nas tabelas da camada Gold, prontas para consumo analítico/BI.

Pré-requisitos:
- Executar `sql/ddl/03_create_gold_tables.sql`.
- Executar o notebook `03_transform_silver.ipynb`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))

from common.spark_utils import get_spark_session
from common.gold_utils import (
    build_dim_tempo,
    build_dim_livro,
    build_dim_categoria,
    build_fato_preco_livro,
)

spark = get_spark_session()
df_silver = spark.table("lakehouse_catalog.silver.livros")

In [ ]:
dim_livro = build_dim_livro(df_silver)
dim_categoria = build_dim_categoria(df_silver)
dim_tempo = build_dim_tempo(df_silver)

dim_livro.write.format("delta").mode("overwrite").saveAsTable("lakehouse_catalog.gold.dim_livro")
dim_categoria.write.format("delta").mode("overwrite").saveAsTable("lakehouse_catalog.gold.dim_categoria")
dim_tempo.write.format("delta").mode("overwrite").saveAsTable("lakehouse_catalog.gold.dim_tempo")

print("Dimensões gravadas com sucesso.")

In [ ]:
dim_livro_gold = spark.table("lakehouse_catalog.gold.dim_livro")
dim_categoria_gold = spark.table("lakehouse_catalog.gold.dim_categoria")
dim_tempo_gold = spark.table("lakehouse_catalog.gold.dim_tempo")

fato_preco_livro = build_fato_preco_livro(
    df_silver, dim_livro_gold, dim_categoria_gold, dim_tempo_gold
)

fato_preco_livro.write.format("delta").mode("overwrite").saveAsTable(
    "lakehouse_catalog.gold.fato_preco_livro"
)

print(f"Registros gravados em gold.fato_preco_livro: {fato_preco_livro.count()}")

## Consulta analítica de exemplo

Preço médio, rating médio e quantidade de livros disponíveis por categoria.

In [ ]:
spark.sql("""
    SELECT
        c.categoria,
        ROUND(AVG(f.preco), 2)  AS preco_medio,
        ROUND(AVG(f.rating), 2) AS rating_medio,
        SUM(CASE WHEN f.disponivel THEN 1 ELSE 0 END) AS qtd_disponiveis
    FROM lakehouse_catalog.gold.fato_preco_livro f
    JOIN lakehouse_catalog.gold.dim_categoria c
      ON f.categoria = c.categoria
    GROUP BY c.categoria
    ORDER BY preco_medio DESC
""").show()